In [1]:
# ============================================================
# SALUD DEL PROYECTO — versión actualizada
# Verifica todos los archivos y hace chequeos básicos
# ============================================================
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Setup ---
RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RAW = RAIZ / 'data' / 'raw'
PROC = RAIZ / 'data' / 'processed'
INF = RAIZ / 'informe'
FIG = INF / 'figuras'

print('=' * 80)
print('🏥 SALUD DEL PROYECTO — ECONOMÍA DEL CONOCIMIENTO LA RIOJA')
print('=' * 80)

# ============================================================
# 1. ARCHIVOS RAW ESENCIALES
# ============================================================
print('\n📁 1. ARCHIVOS RAW ESENCIALES')
print('-' * 80)

RAW_ESENCIALES = {
    'provinciales_serie_empleo_trimestral_4dig_7.xlsx': 'Empleo EC por CIIU 4 dígitos',
    'provinciales_serie_empleo_anual_3.xlsx':           'Empleo por sectores (T5)',
    'provinciales_serie_empresas3_2.xlsx':              'Empresas por CIIU 4 dígitos',
    'provinciales_serie_remuneraciones_mensual_4dig_7.xlsx': 'Salarios por CIIU',
    'SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx':   'Empleo formal mensual',
    'SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx':  'Salario formal mensual',
}

raw_ok = 0
for archivo, desc in RAW_ESENCIALES.items():
    ruta = RAW / archivo
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'   ✅ {archivo:<55} ({size_kb:>7.0f} KB)')
        raw_ok += 1
    else:
        print(f'   ❌ {archivo:<55} FALTA')

print(f'\n   Total: {raw_ok}/{len(RAW_ESENCIALES)} archivos raw OK')

# ============================================================
# 2. ARCHIVOS PROCESSED ACTIVOS
# ============================================================
print('\n📁 2. ARCHIVOS PROCESSED ACTIVOS')
print('-' * 80)

PROC_ESENCIALES = {
    'empleo_ec_2025_trimestral.csv':    'Empleo EC 2025 por CIIU',
    'empleo_ec_por_ciiu_2022_2025.csv': 'Serie Empleo EC 2022-2025',
    'empresas_ec_rioja_2022_2025.xlsx': 'Empresas EC',
    'prima_salarial_ec.xlsx':           'Prima salarial',
    'empleo_formal_ia_mensual.csv':     'Empleo formal mensual',
    'intensidad_provincias.csv':        'Intensidad 9 provincias',
    'Empleo_EC_2025_La_Rioja.xlsx':     'Excel auditable',
}

proc_ok = 0
for archivo, desc in PROC_ESENCIALES.items():
    ruta = PROC / archivo
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'   ✅ {archivo:<45} ({size_kb:>7.1f} KB) — {desc}')
        proc_ok += 1
    else:
        print(f'   ❌ {archivo:<45} FALTA — {desc}')

print(f'\n   Total: {proc_ok}/{len(PROC_ESENCIALES)} archivos processed OK')

# ============================================================
# 3. VERIFICACIÓN DE DATOS CLAVE
# ============================================================
print('\n📊 3. VERIFICACIÓN DE DATOS CLAVE (2025)')
print('-' * 80)

try:
    # --- Empleo EC 2025 ---
    emp = pd.read_csv(PROC / 'empleo_ec_2025_trimestral.csv')
    total_ec = emp['Promedio 2025'].sum()
    
    # Núcleo informático
    nucleo_codes = [7220, 7250, 7290]
    nucleo = emp[emp['CIIU'].isin(nucleo_codes)]['Promedio 2025'].sum()
    
    # Proxies
    proxy_codes = [7421, 7499]
    proxies = emp[emp['CIIU'].isin(proxy_codes)]['Promedio 2025'].sum()
    
    print(f'\n   📌 Empleo EC 2025:')
    print(f'      Total EC (8 CIIU):       {total_ec:>7.1f} puestos   (esperado ~521)')
    print(f'      Núcleo informático:      {nucleo:>7.1f} puestos   (esperado ~63.8)')
    print(f'      Proxies:                 {proxies:>7.1f} puestos   (esperado ~457.6)')
    print(f'      Núcleo como % del EC:    {nucleo/total_ec*100:>7.1f}%         (esperado ~12.2%)')
except Exception as e:
    print(f'   ⚠️  Error al leer empleo: {e}')

try:
    # --- Empresas EC ---
    empre = pd.read_excel(PROC / 'empresas_ec_rioja_2022_2025.xlsx')
    print(f'\n   📌 Empresas EC:')
    print(f'      Archivo:                 {empre.shape[0]} filas × {empre.shape[1]} columnas')
    print(f'      Columnas:                {empre.columns.tolist()[:5]}')
except Exception as e:
    print(f'   ⚠️  Error al leer empresas: {e}')

try:
    # --- Prima salarial ---
    prima = pd.read_excel(PROC / 'prima_salarial_ec.xlsx')
    print(f'\n   📌 Prima salarial:')
    print(f'      Archivo:                 {prima.shape[0]} filas × {prima.shape[1]} columnas')
    print(f'      Columnas:                {prima.columns.tolist()[:5]}')
except Exception as e:
    print(f'   ⚠️  Error al leer prima: {e}')

try:
    # --- Intensidad ---
    inten = pd.read_csv(PROC / 'intensidad_provincias.csv')
    print(f'\n   📌 Intensidad empresarial:')
    print(f'      Provincias:              {len(inten)}')
    print(f'      Columnas:                {inten.columns.tolist()}')
except Exception as e:
    print(f'   ⚠️  Error al leer intensidad: {e}')

# ============================================================
# 4. INFORMES Y DASHBOARD
# ============================================================
print('\n📁 4. INFORMES Y DASHBOARD')
print('-' * 80)

# Informe vigente
informe_qmd = INF / 'informe_empleo_ec.qmd'
if informe_qmd.exists():
    size_kb = informe_qmd.stat().st_size / 1024
    print(f'   ✅ Informe vigente: informe_empleo_ec.qmd ({size_kb:.1f} KB)')
else:
    print(f'   ❌ Informe vigente NO encontrado')

# Figuras usadas
figuras_usadas = [
    'F41_empleo_ec_2025_por_ciiu.png',
    'empleo_ec_2022_2025_por_ciiu.png',
    'F33_salarios_ciiu.png',
    'F34_prima_salarial.png',
    'F36_empleo_share.png',
    'F37_intensidad.png',
]
figs_ok = sum(1 for f in figuras_usadas if (FIG / f).exists())
print(f'   Figuras usadas por el informe: {figs_ok}/{len(figuras_usadas)}')

# Dashboard
dash = RAIZ / 'dashboard' / 'index.html'
if dash.exists():
    size_kb = dash.stat().st_size / 1024
    print(f'   ✅ Dashboard: index.html ({size_kb:.1f} KB)')
else:
    # Buscar en otros lugares
    candidatos = list(RAIZ.rglob('index.html'))
    candidatos = [c for c in candidatos if '.venv' not in str(c)]
    if candidatos:
        print(f'   ⚠️  Dashboard encontrado en: {candidatos[0].relative_to(RAIZ)}')
    else:
        print(f'   ❌ Dashboard NO encontrado')

# ============================================================
# 5. MÓDULOS EN src/
# ============================================================
print('\n📁 5. MÓDULOS EN src/')
print('-' * 80)

SRC = RAIZ / 'src'
modulos_clave = [
    'empresas_ec.py',
    'prima_salarial.py',
    'recaudacion_ec.py',
    'empleo.py',
    'benchmark_intensidad.py',
]

for mod in modulos_clave:
    ruta_mod = SRC / mod
    if ruta_mod.exists():
        size_kb = ruta_mod.stat().st_size / 1024
        print(f'   ✅ {mod:<30} ({size_kb:>6.1f} KB)')
    else:
        print(f'   ❌ {mod:<30} FALTA')

# ============================================================
# 6. RESUMEN FINAL
# ============================================================
print('\n' + '=' * 80)
print('📋 RESUMEN')
print('=' * 80)

total_checks = len(RAW_ESENCIALES) + len(PROC_ESENCIALES) + len(modulos_clave)
total_ok = raw_ok + proc_ok

print(f'\n   Archivos raw:           {raw_ok}/{len(RAW_ESENCIALES)} OK')
print(f'   Archivos processed:     {proc_ok}/{len(PROC_ESENCIALES)} OK')
print(f'   Informe vigente:        {"✅ OK" if informe_qmd.exists() else "❌ FALTA"}')
print(f'   Dashboard:              {"✅ OK" if dash.exists() else "⚠️  Revisar"}')
print(f'   Módulos src/:           {sum(1 for m in modulos_clave if (SRC/m).exists())}/{len(modulos_clave)} OK')

if raw_ok == len(RAW_ESENCIALES) and proc_ok == len(PROC_ESENCIALES):
    print('\n   ✅ PROYECTO SALUDABLE')
else:
    print('\n   ⚠️  Faltan archivos. Revisar arriba.')

print('=' * 80)

🏥 SALUD DEL PROYECTO — ECONOMÍA DEL CONOCIMIENTO LA RIOJA

📁 1. ARCHIVOS RAW ESENCIALES
--------------------------------------------------------------------------------
   ✅ provinciales_serie_empleo_trimestral_4dig_7.xlsx        (   4281 KB)
   ✅ provinciales_serie_empleo_anual_3.xlsx                  (    644 KB)
   ✅ provinciales_serie_empresas3_2.xlsx                     (   1171 KB)
   ✅ provinciales_serie_remuneraciones_mensual_4dig_7.xlsx   (  24470 KB)
   ✅ SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx          (    139 KB)
   ✅ SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx         (    102 KB)

   Total: 6/6 archivos raw OK

📁 2. ARCHIVOS PROCESSED ACTIVOS
--------------------------------------------------------------------------------
   ✅ empleo_ec_2025_trimestral.csv                 (    0.5 KB) — Empleo EC 2025 por CIIU
   ✅ empleo_ec_por_ciiu_2022_2025.csv              (    0.8 KB) — Serie Empleo EC 2022-2025
   ✅ empresas_ec_rioja_2022_2025.xlsx              (    8.3 

In [2]:
# ============================================================
# Verificar ubicación del dashboard
# ============================================================
from pathlib import Path

RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')

print('🔍 BUSCANDO index.html Y CARPETA dashboard/')
print('=' * 80)

# Ver todo lo que hay en dashboard/
dash = RAIZ / 'dashboard'
print(f'\n📁 Contenido de {dash.relative_to(RAIZ)}/:')
if dash.exists():
    for item in sorted(dash.iterdir()):
        if item.is_dir():
            print(f'   📂 {item.name}/')
            # Ver contenido de subcarpetas
            for sub in sorted(item.iterdir())[:10]:
                kb = sub.stat().st_size / 1024 if sub.is_file() else 0
                print(f'      └── {sub.name:<40} {kb:>7.1f} KB')
        else:
            kb = item.stat().st_size / 1024
            print(f'   📄 {item.name:<45} {kb:>7.1f} KB')
else:
    print('   ⚠️  No existe la carpeta dashboard/')

# Buscar TODOS los index.html en el proyecto
print('\n🔍 TODOS los index.html en el proyecto:')
for f in RAIZ.rglob('index.html'):
    if '.venv' not in str(f) and 'site-packages' not in str(f):
        rel = f.relative_to(RAIZ)
        kb = f.stat().st_size / 1024
        print(f'   {rel}  ({kb:.1f} KB)')

🔍 BUSCANDO index.html Y CARPETA dashboard/

📁 Contenido de dashboard/:
   📂 figuras/
      └── DASH1_opex_rioja.png                        73.0 KB
      └── DASH2_opex_benchmark.png                    58.7 KB
      └── DASH3_empleo.png                           106.3 KB
      └── DASH4_salario_real.png                     141.0 KB
      └── DASH5_dgip.png                              72.2 KB
      └── index.html                                  58.3 KB

🔍 TODOS los index.html en el proyecto:
   dashboard\figuras\index.html  (58.3 KB)


In [3]:
# ============================================================
# Mover el dashboard a la ubicación correcta
# ============================================================
import shutil
from pathlib import Path

RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')
DASH = RAIZ / 'dashboard'
origen = DASH / 'figuras' / 'index.html'
destino = DASH / 'index.html'

print('=' * 80)
print('📁 MOVER DASHBOARD')
print('=' * 80)

print(f'\n📍 Ubicación actual: dashboard/figuras/index.html')
print(f'📍 Ubicación destino: dashboard/index.html')

if destino.exists():
    print(f'\n⚠️  Ya existe un index.html en {destino}')
    print('   No se mueve nada para no sobreescribir.')
    print('   Si querés reemplazarlo, borrá primero el destino.')
else:
    # Mover el archivo
    shutil.move(str(origen), str(destino))
    print(f'\n✅ Movido a dashboard/index.html')

# --- Verificar contenido ---
print('\n📊 VERIFICACIÓN DEL DASHBOARD')
print('=' * 80)

if destino.exists():
    contenido = destino.read_text(encoding='utf-8')
    
    # Verificar elementos clave
    checks = {
        'chart-empleo-desglose': 'Gráfico desglose 2025',
        'chart-empleo-ciiu': 'Serie 2022-2025',
        'chart-prima': 'Gráfico de prima',
        'Núcleo informático': 'Núcleo informático (nuevo nombre)',
        'ARCA-DGIP': 'ARCA-DGIP (nuevo nombre)',
        'Highcharts.chart': 'Highcharts',
    }
    
    for clave, desc in checks.items():
        if clave in contenido:
            print(f'   ✅ {desc}')
        else:
            print(f'   ⚠️  {desc} — no encontrado')
    
    # Tamaño
    kb = destino.stat().st_size / 1024
    print(f'\n   Tamaño: {kb:.1f} KB')

# --- Estado final ---
print('\n📁 ESTADO FINAL DE dashboard/')
print('=' * 80)
for item in sorted(DASH.iterdir()):
    if item.is_dir():
        print(f'   📂 {item.name}/')
        for sub in sorted(item.iterdir()):
            kb = sub.stat().st_size / 1024
            print(f'      └── {sub.name:<40} {kb:>7.1f} KB')
    else:
        kb = item.stat().st_size / 1024
        print(f'   📄 {item.name:<45} {kb:>7.1f} KB')

print('\n' + '=' * 80)
print('✅ Dashboard en ubicación correcta: dashboard/index.html')
print('=' * 80)
print('\n💡 Para abrirlo:')
print(f'   - Doble clic en: {destino}')
print('   - O desde PowerShell:')
print(f'     cd "{DASH}"')
print('     start index.html')

📁 MOVER DASHBOARD

📍 Ubicación actual: dashboard/figuras/index.html
📍 Ubicación destino: dashboard/index.html

✅ Movido a dashboard/index.html

📊 VERIFICACIÓN DEL DASHBOARD
   ✅ Gráfico desglose 2025
   ✅ Serie 2022-2025
   ⚠️  Gráfico de prima — no encontrado
   ✅ Núcleo informático (nuevo nombre)
   ✅ ARCA-DGIP (nuevo nombre)
   ✅ Highcharts

   Tamaño: 58.3 KB

📁 ESTADO FINAL DE dashboard/
   📂 figuras/
      └── DASH1_opex_rioja.png                        73.0 KB
      └── DASH2_opex_benchmark.png                    58.7 KB
      └── DASH3_empleo.png                           106.3 KB
      └── DASH4_salario_real.png                     141.0 KB
      └── DASH5_dgip.png                              72.2 KB
   📄 index.html                                       58.3 KB

✅ Dashboard en ubicación correcta: dashboard/index.html

💡 Para abrirlo:
   - Doble clic en: C:\Users\HP\Documents\Economy\economia_c\dashboard\index.html
   - O desde PowerShell:
     cd "C:\Users\HP\Documents\Econom

In [4]:
import shutil
from pathlib import Path

RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')
FIGS = RAIZ / 'dashboard' / 'figuras'
BACKUP = RAIZ / 'dashboard' / '_viejas'
BACKUP.mkdir(exist_ok=True)

movidas = 0
for f in FIGS.glob('*.png'):
    shutil.move(str(f), str(BACKUP / f.name))
    movidas += 1

# Borrar carpeta vacía
try:
    FIGS.rmdir()
    print(f'✅ {movidas} PNGs movidas a dashboard/_viejas/')
    print(f'✅ Carpeta dashboard/figuras/ eliminada')
except OSError as e:
    print(f'✅ {movidas} PNGs movidas a dashboard/_viejas/')
    print(f'⚠️  Carpeta no vacía: {e}')

# Estado final
print('\n📁 dashboard/ final:')
for item in sorted((RAIZ / 'dashboard').iterdir()):
    if item.is_dir():
        print(f'   📂 {item.name}/')
        for sub in sorted(item.iterdir()):
            kb = sub.stat().st_size / 1024
            print(f'      └── {sub.name:<40} {kb:>7.1f} KB')
    else:
        kb = item.stat().st_size / 1024
        print(f'   📄 {item.name:<45} {kb:>7.1f} KB')

✅ 5 PNGs movidas a dashboard/_viejas/
✅ Carpeta dashboard/figuras/ eliminada

📁 dashboard/ final:
   📂 _viejas/
      └── DASH1_opex_rioja.png                        73.0 KB
      └── DASH2_opex_benchmark.png                    58.7 KB
      └── DASH3_empleo.png                           106.3 KB
      └── DASH4_salario_real.png                     141.0 KB
      └── DASH5_dgip.png                              72.2 KB
   📄 index.html                                       58.3 KB


In [5]:
# ============================================================
# INVENTARIO DE DATOS PARA EL INFORME
# Muestra qué datos tenés disponibles por temática
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

RAIZ = Path(r'C:\Users\HP\Documents\Economy\economia_c')
RAW = RAIZ / 'data' / 'raw'
PROC = RAIZ / 'data' / 'processed'

print('=' * 100)
print('📊 INVENTARIO DE DATOS DISPONIBLES PARA EL INFORME')
print('=' * 100)

# ============================================================
# 1. EMPLEO EC (por CIIU)
# ============================================================
print('\n' + '━' * 100)
print('1️⃣  EMPLEO EC — Por CIIU (4 dígitos)')
print('━' * 100)

try:
    df = pd.read_csv(PROC / 'empleo_ec_2025_trimestral.csv')
    print('\n📁 Archivo: empleo_ec_2025_trimestral.csv')
    print(f'   Códigos con datos: {len(df[df["Promedio 2025"]>0])}')
    print(f'   Total EC 2025: {df["Promedio 2025"].sum():.1f} puestos')
    print()
    print(f'   {"CIIU":<6} {"Rama":<45} {"2025":>8} {"Tipo":<20}')
    print('   ' + '-' * 85)
    for _, r in df.iterrows():
        if r['Promedio 2025'] > 0:
            print(f'   {r["CIIU"]:<6} {r["Rama"]:<45} {r["Promedio 2025"]:>8.1f} {"núcleo" if r["CIIU"] in [7220,7250,7290] else "proxy":<20}')
        else:
            print(f'   {r["CIIU"]:<6} {r["Rama"]:<45} {"s/d":>8} {"sin datos":<20}')
    print(f'\n   ✅ DISPONIBLE: Empleo EC total, núcleo informático, desglose por código')
    print(f'   📌 Relevancia para informe: ALTA (dato central del análisis)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 2. SERIE HISTÓRICA EC (2022-2025)
# ============================================================
print('\n' + '━' * 100)
print('2️⃣  EMPLEO EC — Serie histórica 2022-2025')
print('━' * 100)

try:
    df = pd.read_csv(PROC / 'empleo_ec_por_ciiu_2022_2025.csv')
    print('\n📁 Archivo: empleo_ec_por_ciiu_2022_2025.csv')
    print(f'   Años cubiertos: 2022, 2023, 2024, 2025')
    print(f'   Códigos: {len(df)}')
    print()
    print(f'   {"CIIU":<6} {"Rama":<35} {"2022":>7} {"2023":>7} {"2024":>7} {"2025":>7} {"Δ%":>8}')
    print('   ' + '-' * 90)
    for _, r in df.iterrows():
        v22 = f'{r["2022"]:.1f}' if pd.notna(r['2022']) else '—'
        v23 = f'{r["2023"]:.1f}' if pd.notna(r['2023']) else '—'
        v24 = f'{r["2024"]:.1f}' if pd.notna(r['2024']) else '—'
        v25 = f'{r["2025"]:.1f}' if pd.notna(r['2025']) else '—'
        d = f'{r["Δ% 22→25"]:+.1f}%' if pd.notna(r.get('Δ% 22→25')) else '—'
        print(f'   {r["CIIU"]:<6} {r["Rama"][:35]:<35} {v22:>7} {v23:>7} {v24:>7} {v25:>7} {d:>8}')
    print(f'\n   ✅ DISPONIBLE: Serie completa por código y total')
    print(f'   📌 Relevancia para informe: ALTA (tendencia y dinámica)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 3. EMPRESAS EC
# ============================================================
print('\n' + '━' * 100)
print('3️⃣  EMPRESAS EC')
print('━' * 100)

try:
    df = pd.read_excel(PROC / 'empresas_ec_rioja_2022_2025.xlsx')
    print('\n📁 Archivo: empresas_ec_rioja_2022_2025.xlsx')
    print(f'   Columnas: {df.columns.tolist()}')
    print()
    print(df.to_string(index=False))
    print(f'\n   ✅ DISPONIBLE: Total empresas EC por año y share provincial')
    print(f'   📌 Relevancia para informe: ALTA (masa crítica empresarial)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 4. PRIMA SALARIAL
# ============================================================
print('\n' + '━' * 100)
print('4️⃣  PRIMA SALARIAL EC')
print('━' * 100)

try:
    df = pd.read_excel(PROC / 'prima_salarial_ec.xlsx')
    print('\n📁 Archivo: prima_salarial_ec.xlsx')
    print(f'   Columnas: {df.columns.tolist()}')
    print()
    print(df.to_string(index=False))
    print(f'\n   ✅ DISPONIBLE: Prima por 3 definiciones (núcleo, amplio, completo)')
    print(f'   📌 Relevancia para informe: ALTA (diferencial salarial)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 5. INTENSIDAD PROVINCIAL (BENCHMARK)
# ============================================================
print('\n' + '━' * 100)
print('5️⃣  INTENSIDAD EMPRESARIAL EC — Benchmark provincial')
print('━' * 100)

try:
    df = pd.read_csv(PROC / 'intensidad_provincias.csv')
    print('\n📁 Archivo: intensidad_provincias.csv')
    print(f'   Provincias: {len(df)}')
    print()
    print(df.to_string(index=False))
    print(f'\n   ✅ DISPONIBLE: Comparación con 8 provincias + La Rioja')
    print(f'   📌 Relevancia para informe: ALTA (posicionamiento federal)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 6. EMPLEO FORMAL MENSUAL
# ============================================================
print('\n' + '━' * 100)
print('6️⃣  EMPLEO FORMAL PRIVADO — Serie mensual')
print('━' * 100)

try:
    df = pd.read_csv(PROC / 'empleo_formal_ia_mensual.csv', parse_dates=['fecha'])
    print('\n📁 Archivo: empleo_formal_ia_mensual.csv')
    print(f'   Rango: {df["fecha"].min().strftime("%b-%Y")} → {df["fecha"].max().strftime("%b-%Y")}')
    print(f'   Filas: {len(df)}')
    print(f'   Últimos 3 meses:')
    print(df.tail(3).to_string(index=False))
    print(f'\n   ✅ DISPONIBLE: Serie mensual con i.a. y tendencia')
    print(f'   📌 Relevancia para informe: MEDIA (contexto macro)')
except Exception as e:
    print(f'   ❌ Error: {e}')

# ============================================================
# 7. RECAUDACIÓN DGIP
# ============================================================
print('\n' + '━' * 100)
print('7️⃣  RECAUDACIÓN EC (DGIP)')
print('━' * 100)

# Buscar archivo
rec_file = RAIZ / 'data' / 'processed' / '_backup' / 'recaudacion_ec_rioja.xlsx'
if not rec_file.exists():
    rec_file = PROC / 'recaudacion_ec_rioja.xlsx'

if rec_file.exists():
    try:
        df = pd.read_excel(rec_file)
        print(f'\n📁 Archivo: {rec_file.name}')
        print(f'   Filas: {len(df)}')
        print(f'   Columnas: {df.columns.tolist()}')
        print()
        print(df.to_string(index=False))
        print(f'\n   ✅ DISPONIBLE: Recaudación EC por año')
        print(f'   📌 Relevancia para informe: ALTA (impacto fiscal)')
    except Exception as e:
        print(f'   ❌ Error: {e}')
else:
    print('\n   ⚠️  Archivo no encontrado en data/processed/')
    print('   Buscando en _backup/...')
    for f in (RAIZ / 'data' / 'processed' / '_backup').glob('recaud*'):
        print(f'   Encontrado: {f.name}')

# ============================================================
# 8. CONTEXTO MACRO (VAB, I+D, conectividad, exportaciones)
# ============================================================
print('\n' + '━' * 100)
print('8️⃣  CONTEXTO MACRO')
print('━' * 100)

CONTEXTO = {
    'VAB provincial 2025': '-25,7% i.a. (sh_opex_*.xls)',
    'Inversión I+D 2024': '$9.076 M (+425% vs 2022) (actividad_ied.xlsx)',
    'Investigadores EJC 2024': '274 (pico 2013: 549) (investigadores_por_nivel.csv)',
    'Exportaciones 2024': 'USD 219 M (3,7% alta tech) (INDEC)',
    'Fibra óptica Q1-2026': '27.564 accesos (+188,7%) (ENACOM)',
}
for k, v in CONTEXTO.items():
    print(f'   • {k:<30} {v}')
print(f'\n   ✅ DISPONIBLE: Contexto macro para sección final')
print(f'   📌 Relevancia para informe: MEDIA (complementa el análisis)')

# ============================================================
# RESUMEN PARA DECIDIR
# ============================================================
print('\n' + '=' * 100)
print('📋 RESUMEN — QUÉ TENÉS PARA EL INFORME')
print('=' * 100)

RESUMEN = """
┌─────────────────────────────────────────────────────────────────────────────┐
│  TEMÁTICA                        DATOS              RELEVANCIA   ¿INCLUIR?  │
├─────────────────────────────────────────────────────────────────────────────┤
│  1. Empleo EC 2025               521 puestos        ALTA         ✅ Sí      │
│  2. Núcleo informático 2025      63,8 puestos       ALTA         ✅ Sí      │
│  3. Desglose por CIIU            5 códigos          ALTA         ✅ Sí      │
│  4. Serie 2022-2025              4 años             ALTA         ✅ Sí      │
│  5. Empresas EC (2022-2025)      95→84              ALTA         ✅ Sí      │
│  6. Prima salarial (3 defs.)     +29%/+8,4%         ALTA         ✅ Sí      │
│  7. Intensidad provincial        9 provincias       ALTA         ✅ Sí      │
│  8. Recaudación DGIP             ×32,9              ALTA         ✅ Sí      │
│  9. Empleo formal mensual        hasta jun-2026     MEDIA        ⚠️  Quizás  │
│ 10. Salario por CIIU             $3,1M (7290)       ALTA         ✅ Sí      │
│ 11. VAB provincial               -25,7%             MEDIA        ⚠️  Quizás  │
│ 12. I+D                          $9.076 M           BAJA         ❌ No      │
│ 13. Exportaciones                USD 219 M          BAJA         ❌ No      │
│ 14. Conectividad (fibra)         +188,7%            BAJA         ❌ No      │
│ 15. Contexto macro general       varios              BAJA         ❌ No      │
└─────────────────────────────────────────────────────────────────────────────┘

SUGERENCIA DE ESTRUCTURA (informe monográfico de empleo):
  1. Resumen ejecutivo                      — usa datos 1-4, 6
  2. Definición y metodología               — contexto general
  3. Radiografía del sector                 — datos 1, 2, 3, 5, 8
  4. Serie histórica 2022-2025              — datos 4, 9
  5. Salarios y prima                       — datos 6, 10
  6. Comparación federal                    — dato 7
  7. Conclusiones                           — síntesis

  El contexto (VAB, I+D, exportaciones, conectividad) puede ir como
  ANEXO o directamente OMITIRSE si el foco es empleo.
"""
print(RESUMEN)

📊 INVENTARIO DE DATOS DISPONIBLES PARA EL INFORME

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1️⃣  EMPLEO EC — Por CIIU (4 dígitos)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📁 Archivo: empleo_ec_2025_trimestral.csv
   Códigos con datos: 5
   Total EC 2025: 521.4 puestos

   CIIU   Rama                                              2025 Tipo                
   -------------------------------------------------------------------------------------
   7210   Consultores en equipo de informática               s/d sin datos           
   7220   Consultores en informática                        26.8 núcleo              
   7250   Mantenimiento y reparación informática             4.2 núcleo              
   7290   Informática n.c.p.                                32.8 núcleo              
   7300   Investigación y desarrollo                         s/d sin datos           
   74

In [3]:
import os
from pathlib import Path

os.chdir(r'C:\Users\HP\Documents\Economy\economia_c')
print("Directorio actual:", Path.cwd())

Directorio actual: C:\Users\HP\Documents\Economy\economia_c


In [4]:
import pandas as pd

RUTA = Path.cwd() / "data" / "raw" / "provinciales_serie_empresas3_2.xlsx"
df = pd.read_excel(RUTA, sheet_name="La Rioja", header=None)

print(f"Archivo: {RUTA.name}")
print(f"Shape: {df.shape}")
print()
print("Título (fila 2):")
print(df.iloc[2, 0])

Archivo: provinciales_serie_empresas3_2.xlsx
Shape: (310, 31)

Título (fila 2):
Empresas del sector privado. Según rama de actividad a 4 dígitos del CIIU rev. 3. Activas al cierre de cada año. Anual 1996-2024.


In [5]:
header = df.iloc[6, :].tolist()
anios = [h for h in header if isinstance(h, (int, float)) and 1990 < h < 2100]
print(f"Años disponibles: {[int(a) for a in anios]}")

Años disponibles: [1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025]


In [6]:
totales_provinciales = {}
for i, h in enumerate(header):
    if h in (2022, 2024, 2025):
        total = pd.to_numeric(df.iloc[7:, i], errors="coerce").sum()
        totales_provinciales[int(h)] = int(total)
        print(f"  {int(h)}: {int(total)} empresas totales")

  2022: 2873 empresas totales
  2024: 3255 empresas totales
  2025: 2935 empresas totales


In [7]:
CIIU_EC = [7210, 7220, 7250, 7290, 7300, 7421, 7430, 7499]

empresas_ec = {}
for i, h in enumerate(header):
    if h in (2022, 2024, 2025):
        total = 0
        detalle = []
        for idx in range(7, len(df)):
            rama = df.iloc[idx, 0]
            try:
                rama_int = int(rama) if pd.notna(rama) else None
            except (ValueError, TypeError):
                continue
            if rama_int in CIIU_EC:
                val = pd.to_numeric(df.iloc[idx, i], errors="coerce")
                if pd.notna(val):
                    total += int(val)
                    detalle.append(f"{rama_int}: {int(val)}")
        empresas_ec[int(h)] = total
        print(f"  {int(h)}: {total} empresas EC")
        print(f"     Desglose: {' | '.join(detalle)}")
        print()

  2022: 95 empresas EC
     Desglose: 7220: 12 | 7290: 13 | 7421: 26 | 7430: 4 | 7499: 40

  2024: 102 empresas EC
     Desglose: 7220: 11 | 7250: 4 | 7290: 13 | 7421: 31 | 7430: 5 | 7499: 38

  2025: 84 empresas EC
     Desglose: 7220: 9 | 7250: 4 | 7290: 12 | 7421: 28 | 7499: 31

